# Iterative Burst Detector Synthetic Validation

This notebook generates synthetic spike-train cultures with known ground-truth burst and silence intervals, runs the iterative burst detector, and visualizes whether detected events match the simulated behavior.

Success criteria:
- Independent Poisson cultures should produce few or no network bursts.
- Cascade-propagation cultures should produce network bursts near injected burst epochs.
- Unit count, burst length, burst timing, recruitment strength, and burst morphology should vary across simulated cultures.
- Composite cultures with multiple burst types should expose detector behavior when one well contains heterogeneous network events.
- Quantification tables should report detection precision/recall/F1, temporal overlap, timing error, and detector quality columns such as LLR, composite score, and FF.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
import sys
import os
import tempfile

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib-cache"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Make imports work whether the notebook is launched from the repo root or notebooks/.
for candidate in (Path.cwd(), Path.cwd().parent):
    if (candidate / "pipeline_tasks").exists():
        repo_root = candidate
        break
else:
    raise RuntimeError("Could not locate repository root containing pipeline_tasks/")

if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from pipeline_tasks.analysis import (
    BurstDetectorConfig,
    IterativeBurstConfig,
    compute_iterative_bursts,
    compute_network_bursts,
)
from pipeline_tasks.analysis.burst_detector import BurstDetectorError
from pipeline_tasks.analysis.iterative_burst_detector import IterativeBurstError

plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
})

BASE_SEED = 20260510
DEFAULT_DURATION_S = 150.0
N_UNITS = 24

# Manually set this directory before running the notebook.
# All generated figures are saved here as PNG files.
FIGURE_OUTPUT_DIR = repo_root / "output" / "iterative_burst_detector_figures"
FIGURE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
print(f"Figures will be saved to: {FIGURE_OUTPUT_DIR}")

## Synthetic Data Model

Independent datasets use per-unit Poisson spike trains only. Connected datasets include cascade, synchronous, long-wave, partial-recruitment, doublet, clustered, irregular, and post-burst-silence morphologies. Several datasets randomize unit count and burst parameters with fixed seeds for reproducibility. The composite culture intentionally combines many burst types in one recording so the detector must handle heterogeneous event shapes within the same well.


In [ ]:
@dataclass
class SyntheticDataset:
    name: str
    culture_type: str
    description: str
    spike_times: dict[str, np.ndarray]
    duration_s: float
    burst_intervals: list[tuple[float, float]]
    silence_intervals: list[tuple[float, float]]
    params: dict


def merge_intervals(intervals: list[tuple[float, float]], duration_s: float) -> list[tuple[float, float]]:
    clipped = sorted((max(0.0, s), min(duration_s, e)) for s, e in intervals if e > 0 and s < duration_s)
    if not clipped:
        return []
    merged = [clipped[0]]
    for s, e in clipped[1:]:
        ps, pe = merged[-1]
        if s <= pe:
            merged[-1] = (ps, max(pe, e))
        else:
            merged.append((s, e))
    return merged


def complement_intervals(duration_s: float, blocked: list[tuple[float, float]]) -> list[tuple[float, float]]:
    blocked = merge_intervals(blocked, duration_s)
    allowed = []
    cursor = 0.0
    for s, e in blocked:
        if s > cursor:
            allowed.append((cursor, s))
        cursor = max(cursor, e)
    if cursor < duration_s:
        allowed.append((cursor, duration_s))
    return allowed


def poisson_spikes_in_intervals(rate_hz: float, intervals: list[tuple[float, float]], rng: np.random.Generator) -> np.ndarray:
    if rate_hz <= 0 or not intervals:
        return np.array([], dtype=float)
    pieces = []
    for s, e in intervals:
        n = rng.poisson(rate_hz * max(0.0, e - s))
        if n:
            pieces.append(rng.uniform(s, e, n))
    if not pieces:
        return np.array([], dtype=float)
    return np.sort(np.concatenate(pieces).astype(float))


def make_unit_ids(n_units: int) -> list[str]:
    return [f"unit_{i:02d}" for i in range(n_units)]


def as_per_burst_values(value: float | list[float], n_bursts: int) -> list[float]:
    if isinstance(value, list):
        if len(value) != n_bursts:
            raise ValueError(f"Expected {n_bursts} per-burst values, got {len(value)}")
        return [float(v) for v in value]
    return [float(value)] * n_bursts


def generate_independent_poisson(
    name: str,
    description: str,
    rates_hz: np.ndarray,
    duration_s: float,
    seed: int,
) -> SyntheticDataset:
    rng = np.random.default_rng(seed)
    units = make_unit_ids(len(rates_hz))
    full_interval = [(0.0, duration_s)]
    spike_times = {
        unit: poisson_spikes_in_intervals(float(rate), full_interval, rng)
        for unit, rate in zip(units, rates_hz)
    }
    return SyntheticDataset(
        name=name,
        culture_type="independent_poisson",
        description=description,
        spike_times=spike_times,
        duration_s=duration_s,
        burst_intervals=[],
        silence_intervals=[],
        params={"rates_hz": rates_hz.round(3).tolist()},
    )


def generate_cascade_culture(
    name: str,
    description: str,
    baseline_rates_hz: np.ndarray,
    burst_starts_s: list[float],
    burst_rate_hz: float,
    recruitment_probability: float,
    duration_s: float,
    seed: int,
    group_count: int = 4,
    group_delay_s: float = 0.08,
    group_burst_duration_s: float | list[float] = 0.22,
    burst_jitter_s: float | None = None,
    silence_after_s: float = 0.0,
    silence_rate_scale: float = 0.05,
) -> SyntheticDataset:
    rng = np.random.default_rng(seed)
    units = make_unit_ids(len(baseline_rates_hz))
    unit_groups = np.array_split(np.arange(len(units)), group_count)
    burst_durations = as_per_burst_values(group_burst_duration_s, len(burst_starts_s))

    cascade_spans = [(group_count - 1) * group_delay_s + dur for dur in burst_durations]
    burst_intervals = [(start, start + span) for start, span in zip(burst_starts_s, cascade_spans)]
    silence_intervals = [
        (start + span, start + span + silence_after_s)
        for start, span in zip(burst_starts_s, cascade_spans)
        if silence_after_s > 0
    ]
    blocked = merge_intervals(burst_intervals + silence_intervals, duration_s)
    background_intervals = complement_intervals(duration_s, blocked)

    spike_times: dict[str, list[float]] = {unit: [] for unit in units}

    # Independent background outside burst and silence windows.
    for unit, rate in zip(units, baseline_rates_hz):
        spike_times[unit].extend(poisson_spikes_in_intervals(float(rate), background_intervals, rng).tolist())

    # Suppressed post-burst activity for silence datasets.
    if silence_intervals:
        for unit, rate in zip(units, baseline_rates_hz):
            spike_times[unit].extend(
                poisson_spikes_in_intervals(float(rate) * silence_rate_scale, silence_intervals, rng).tolist()
            )

    # Cascade bursts: each group is recruited after a fixed delay.
    for burst_start, burst_duration in zip(burst_starts_s, burst_durations):
        jitter = float(burst_jitter_s) if burst_jitter_s is not None else max(0.008, burst_duration / 7)
        for group_idx, group in enumerate(unit_groups):
            group_start = burst_start + group_idx * group_delay_s
            group_end = group_start + burst_duration
            for unit_idx in group:
                if rng.random() > recruitment_probability:
                    continue
                unit = units[int(unit_idx)]
                n_spikes = rng.poisson(burst_rate_hz * burst_duration)
                n_spikes = max(1, int(n_spikes))
                burst_spikes = rng.normal(
                    loc=(group_start + group_end) / 2,
                    scale=jitter,
                    size=n_spikes,
                )
                burst_spikes = burst_spikes[(burst_spikes >= group_start) & (burst_spikes < group_end)]
                if burst_spikes.size == 0:
                    burst_spikes = np.array([rng.uniform(group_start, group_end)])
                spike_times[unit].extend(burst_spikes.tolist())

    sorted_spikes = {
        unit: np.sort(np.asarray([s for s in spikes if 0.0 <= s < duration_s], dtype=float))
        for unit, spikes in spike_times.items()
    }
    return SyntheticDataset(
        name=name,
        culture_type="cascade_connected",
        description=description,
        spike_times=sorted_spikes,
        duration_s=duration_s,
        burst_intervals=merge_intervals(burst_intervals, duration_s),
        silence_intervals=merge_intervals(silence_intervals, duration_s),
        params={
            "baseline_rate_mean_hz": float(np.mean(baseline_rates_hz)),
            "baseline_rate_min_hz": float(np.min(baseline_rates_hz)),
            "baseline_rate_max_hz": float(np.max(baseline_rates_hz)),
            "burst_rate_hz": burst_rate_hz,
            "recruitment_probability": recruitment_probability,
            "n_bursts": len(burst_starts_s),
            "group_delay_s": group_delay_s,
            "group_burst_duration_s": burst_durations,
            "silence_after_s": silence_after_s,
            "silence_rate_scale": silence_rate_scale,
        },
    )


def generate_composite_burst_culture(
    name: str,
    description: str,
    n_units: int,
    baseline_rates_hz: np.ndarray,
    burst_specs: list[dict],
    duration_s: float,
    seed: int,
) -> SyntheticDataset:
    """Generate a single culture containing several burst morphologies.

    Supported burst types:
    - synchronous: all recruited units share one compact burst window.
    - cascade: recruited groups activate sequentially.
    - long_wave: slower cascade with longer group activation.
    - partial: only a lower-probability subnetwork is recruited.
    - doublet: two compact synchronous bursts inside one ground-truth event.
    """
    rng = np.random.default_rng(seed)
    units = make_unit_ids(n_units)
    unit_indices = np.arange(n_units)
    spike_times: dict[str, list[float]] = {unit: [] for unit in units}

    truth_intervals: list[tuple[float, float]] = []
    silence_intervals: list[tuple[float, float]] = []
    for spec in burst_specs:
        start = float(spec["start_s"])
        burst_type = spec["type"]
        if burst_type == "doublet":
            duration = float(spec.get("duration_s", 0.12))
            separation = float(spec.get("separation_s", 0.28))
            end = start + separation + duration
        elif burst_type in {"cascade", "long_wave"}:
            group_count = int(spec.get("group_count", 4))
            group_delay = float(spec.get("group_delay_s", 0.08))
            duration = float(spec.get("duration_s", 0.22))
            end = start + (group_count - 1) * group_delay + duration
        else:
            duration = float(spec.get("duration_s", 0.22))
            end = start + duration
        truth_intervals.append((start, end))
        silence_after = float(spec.get("silence_after_s", 0.0))
        if silence_after > 0:
            silence_intervals.append((end, end + silence_after))

    blocked = merge_intervals(truth_intervals + silence_intervals, duration_s)
    background_intervals = complement_intervals(duration_s, blocked)

    for unit, rate in zip(units, baseline_rates_hz):
        spike_times[unit].extend(poisson_spikes_in_intervals(float(rate), background_intervals, rng).tolist())

    if silence_intervals:
        silence_scale = 0.03
        for unit, rate in zip(units, baseline_rates_hz):
            spike_times[unit].extend(
                poisson_spikes_in_intervals(float(rate) * silence_scale, silence_intervals, rng).tolist()
            )

    def add_window_spikes(indices: np.ndarray, start: float, end: float, rate_hz: float, recruitment: float) -> None:
        duration = max(0.0, end - start)
        jitter = max(0.006, duration / 7)
        for idx in indices:
            if rng.random() > recruitment:
                continue
            n_spikes = max(1, int(rng.poisson(rate_hz * duration)))
            spikes = rng.normal(loc=(start + end) / 2, scale=jitter, size=n_spikes)
            spikes = spikes[(spikes >= start) & (spikes < end)]
            if spikes.size == 0:
                spikes = np.array([rng.uniform(start, end)])
            spike_times[units[int(idx)]].extend(spikes.tolist())

    burst_types_used = []
    for spec in burst_specs:
        burst_type = spec["type"]
        burst_types_used.append(burst_type)
        start = float(spec["start_s"])
        rate = float(spec.get("burst_rate_hz", 55.0))
        recruitment = float(spec.get("recruitment_probability", 0.8))
        duration = float(spec.get("duration_s", 0.22))

        if burst_type == "synchronous":
            add_window_spikes(unit_indices, start, start + duration, rate, recruitment)
        elif burst_type == "partial":
            subnetwork_frac = float(spec.get("subnetwork_frac", 0.45))
            n_sub = max(1, int(round(n_units * subnetwork_frac)))
            subnetwork = rng.choice(unit_indices, size=n_sub, replace=False)
            add_window_spikes(subnetwork, start, start + duration, rate, recruitment)
        elif burst_type == "doublet":
            separation = float(spec.get("separation_s", 0.28))
            add_window_spikes(unit_indices, start, start + duration, rate, recruitment)
            add_window_spikes(unit_indices, start + separation, start + separation + duration, rate, recruitment)
        elif burst_type in {"cascade", "long_wave"}:
            group_count = int(spec.get("group_count", 4))
            group_delay = float(spec.get("group_delay_s", 0.08 if burst_type == "cascade" else 0.18))
            groups = np.array_split(unit_indices, group_count)
            for group_idx, group in enumerate(groups):
                group_start = start + group_idx * group_delay
                add_window_spikes(group, group_start, group_start + duration, rate, recruitment)
        else:
            raise ValueError(f"Unknown burst type: {burst_type}")

    sorted_spikes = {
        unit: np.sort(np.asarray([s for s in spikes if 0.0 <= s < duration_s], dtype=float))
        for unit, spikes in spike_times.items()
    }
    return SyntheticDataset(
        name=name,
        culture_type="composite_connected",
        description=description,
        spike_times=sorted_spikes,
        duration_s=duration_s,
        burst_intervals=merge_intervals(truth_intervals, duration_s),
        silence_intervals=merge_intervals(silence_intervals, duration_s),
        params={
            "n_units": n_units,
            "baseline_rate_mean_hz": float(np.mean(baseline_rates_hz)),
            "baseline_rate_min_hz": float(np.min(baseline_rates_hz)),
            "baseline_rate_max_hz": float(np.max(baseline_rates_hz)),
            "burst_types": burst_types_used,
            "burst_specs": burst_specs,
        },
    )



In [ ]:
def build_synthetic_datasets() -> list[SyntheticDataset]:
    rng = np.random.default_rng(BASE_SEED)
    duration_s = DEFAULT_DURATION_S
    composite_n_units = int(rng.integers(30, 43))

    datasets = [
        generate_independent_poisson(
            "random-low-independent",
            "Independent low-rate Poisson units; expected to be mostly quiet with no network structure.",
            np.full(N_UNITS, 0.15),
            duration_s,
            seed=BASE_SEED + 1,
        ),
        generate_independent_poisson(
            "random-medium-independent",
            "Independent medium-rate Poisson units; random coincidences may occur but no shared drive exists.",
            np.full(N_UNITS, 0.8),
            duration_s,
            seed=BASE_SEED + 2,
        ),
        generate_independent_poisson(
            "random-high-independent",
            "Independent high-rate Poisson units; tests false positives under dense unstructured firing.",
            np.full(N_UNITS, 3.0),
            duration_s,
            seed=BASE_SEED + 3,
        ),
        generate_independent_poisson(
            "random-mixed-heterogeneous",
            "Independent heterogeneous Poisson units spanning low to high individual rates.",
            np.exp(rng.uniform(np.log(0.05), np.log(5.0), N_UNITS)),
            duration_s,
            seed=BASE_SEED + 4,
        ),
        generate_cascade_culture(
            "network-low-sparse-cascade",
            "Few weak cascade bursts on low background firing.",
            np.full(N_UNITS, 0.15),
            [30.0, 78.0, 126.0],
            burst_rate_hz=28.0,
            recruitment_probability=0.55,
            duration_s=duration_s,
            seed=BASE_SEED + 5,
        ),
        generate_cascade_culture(
            "network-high-sparse-cascade",
            "Few cascade bursts on higher individual background rates.",
            np.full(N_UNITS, 1.4),
            [30.0, 78.0, 126.0],
            burst_rate_hz=45.0,
            recruitment_probability=0.65,
            duration_s=duration_s,
            seed=BASE_SEED + 6,
        ),
        generate_cascade_culture(
            "network-medium-frequent-cascade",
            "Moderately frequent cascade bursts with medium individual firing rates.",
            np.full(N_UNITS, 0.6),
            [18.0, 42.0, 66.0, 90.0, 114.0, 138.0],
            burst_rate_hz=48.0,
            recruitment_probability=0.75,
            duration_s=duration_s,
            seed=BASE_SEED + 7,
        ),
        generate_cascade_culture(
            "network-mixed-strong-cascade",
            "Strong frequent cascade bursts with heterogeneous unit baselines.",
            np.exp(rng.uniform(np.log(0.08), np.log(2.5), N_UNITS)),
            [16.0, 34.0, 52.0, 70.0, 88.0, 106.0, 124.0, 142.0],
            burst_rate_hz=70.0,
            recruitment_probability=0.90,
            duration_s=duration_s,
            seed=BASE_SEED + 8,
        ),
        generate_cascade_culture(
            "network-medium-post-burst-silence",
            "Cascade bursts followed by a brief low-rate silence period across connected units.",
            np.full(N_UNITS, 0.8),
            [24.0, 58.0, 92.0, 126.0],
            burst_rate_hz=60.0,
            recruitment_probability=0.85,
            duration_s=duration_s,
            seed=BASE_SEED + 9,
            silence_after_s=1.2,
            silence_rate_scale=0.03,
        ),
        generate_cascade_culture(
            "network-medium-short-duration",
            "Very short cascade bursts; tests temporal resolution and missed short events.",
            np.full(N_UNITS, 0.7),
            [20.0, 45.0, 70.0, 95.0, 120.0],
            burst_rate_hz=95.0,
            recruitment_probability=0.85,
            duration_s=duration_s,
            seed=BASE_SEED + 10,
            group_delay_s=0.025,
            group_burst_duration_s=0.08,
        ),
        generate_cascade_culture(
            "network-medium-long-duration",
            "Long cascade bursts; tests duration recovery and over-trimming.",
            np.full(N_UNITS, 0.7),
            [20.0, 55.0, 90.0, 125.0],
            burst_rate_hz=32.0,
            recruitment_probability=0.90,
            duration_s=duration_s,
            seed=BASE_SEED + 11,
            group_delay_s=0.12,
            group_burst_duration_s=0.65,
        ),
        generate_cascade_culture(
            "network-medium-variable-duration",
            "Same culture with burst durations ranging from brief to long.",
            np.full(N_UNITS, 0.7),
            [18.0, 45.0, 72.0, 99.0, 126.0],
            burst_rate_hz=55.0,
            recruitment_probability=0.82,
            duration_s=duration_s,
            seed=BASE_SEED + 12,
            group_delay_s=0.07,
            group_burst_duration_s=[0.10, 0.18, 0.32, 0.55, 0.80],
        ),
        generate_cascade_culture(
            "network-low-weak-recruitment",
            "Low firing rate and weak recruitment; intentionally difficult positive control.",
            np.full(N_UNITS, 0.2),
            [24.0, 62.0, 100.0, 138.0],
            burst_rate_hz=35.0,
            recruitment_probability=0.38,
            duration_s=duration_s,
            seed=BASE_SEED + 13,
            group_delay_s=0.08,
            group_burst_duration_s=0.25,
        ),
        generate_cascade_culture(
            "network-medium-clustered-bursts",
            "Bursts arrive in close clusters; tests whether nearby bursts are split or merged.",
            np.full(N_UNITS, 0.6),
            [20.0, 20.9, 21.8, 78.0, 78.9, 79.8, 132.0, 132.9, 133.8],
            burst_rate_hz=52.0,
            recruitment_probability=0.78,
            duration_s=duration_s,
            seed=BASE_SEED + 14,
            group_delay_s=0.06,
            group_burst_duration_s=0.22,
        ),
        generate_cascade_culture(
            "network-high-irregular-intervals",
            "High-rate bursts at irregular intervals; tests adaptive background estimation.",
            np.full(N_UNITS, 1.4),
            [12.0, 27.5, 63.0, 74.0, 119.5, 146.0],
            burst_rate_hz=55.0,
            recruitment_probability=0.80,
            duration_s=duration_s,
            seed=BASE_SEED + 15,
            group_delay_s=0.09,
            group_burst_duration_s=0.28,
        ),

        generate_independent_poisson(
            "random-low-few-units",
            "Independent Poisson control with randomized low unit count.",
            np.full(int(rng.integers(8, 13)), 0.35),
            duration_s,
            seed=BASE_SEED + 16,
        ),
        generate_independent_poisson(
            "random-medium-many-units",
            "Independent Poisson control with randomized high unit count.",
            np.full(int(rng.integers(48, 65)), 0.8),
            duration_s,
            seed=BASE_SEED + 17,
        ),
        generate_cascade_culture(
            "network-medium-randomized-cascade-a",
            "Randomized unit count and burst parameters sampled from broad but plausible ranges.",
            np.full(int(rng.integers(16, 33)), float(rng.uniform(0.45, 0.9))),
            sorted(rng.uniform(14.0, 140.0, int(rng.integers(4, 8))).round(1).tolist()),
            burst_rate_hz=float(rng.uniform(42.0, 72.0)),
            recruitment_probability=float(rng.uniform(0.62, 0.90)),
            duration_s=duration_s,
            seed=BASE_SEED + 18,
            group_delay_s=float(rng.uniform(0.04, 0.14)),
            group_burst_duration_s=float(rng.uniform(0.12, 0.42)),
        ),
        generate_cascade_culture(
            "network-high-randomized-cascade-b",
            "Second randomized connected culture with different unit count and morphology.",
            np.full(int(rng.integers(28, 49)), float(rng.uniform(0.9, 1.8))),
            sorted(rng.uniform(12.0, 145.0, int(rng.integers(5, 9))).round(1).tolist()),
            burst_rate_hz=float(rng.uniform(45.0, 80.0)),
            recruitment_probability=float(rng.uniform(0.55, 0.86)),
            duration_s=duration_s,
            seed=BASE_SEED + 19,
            group_delay_s=float(rng.uniform(0.05, 0.18)),
            group_burst_duration_s=float(rng.uniform(0.18, 0.55)),
        ),
        generate_composite_burst_culture(
            "composite-mixed-many-burst-types",
            "One culture containing synchronous, cascade, long-wave, partial, doublet, and post-silence bursts.",
            n_units=composite_n_units,
            baseline_rates_hz=np.exp(rng.uniform(np.log(0.12), np.log(1.6), composite_n_units)),
            burst_specs=[
                {"type": "synchronous", "start_s": 14.0, "duration_s": 0.16, "burst_rate_hz": 78.0, "recruitment_probability": 0.86},
                {"type": "cascade", "start_s": 34.0, "duration_s": 0.24, "group_delay_s": 0.08, "burst_rate_hz": 58.0, "recruitment_probability": 0.78},
                {"type": "long_wave", "start_s": 58.0, "duration_s": 0.62, "group_delay_s": 0.17, "burst_rate_hz": 34.0, "recruitment_probability": 0.86},
                {"type": "partial", "start_s": 86.0, "duration_s": 0.28, "burst_rate_hz": 70.0, "recruitment_probability": 0.82, "subnetwork_frac": 0.42},
                {"type": "doublet", "start_s": 111.0, "duration_s": 0.10, "separation_s": 0.34, "burst_rate_hz": 88.0, "recruitment_probability": 0.78},
                {"type": "cascade", "start_s": 136.0, "duration_s": 0.24, "group_delay_s": 0.07, "burst_rate_hz": 62.0, "recruitment_probability": 0.84, "silence_after_s": 1.0},
            ],
            duration_s=duration_s,
            seed=BASE_SEED + 20,
        ),
    ]
    return datasets


datasets = build_synthetic_datasets()

pd.DataFrame([
    {
        "dataset": d.name,
        "type": d.culture_type,
        "units": len(d.spike_times),
        "duration_s": d.duration_s,
        "truth_bursts": len(d.burst_intervals),
        "truth_silences": len(d.silence_intervals),
        "mean_truth_duration_s": np.mean([e - s for s, e in d.burst_intervals]) if d.burst_intervals else 0.0,
        "mean_spikes_per_unit": np.mean([len(v) for v in d.spike_times.values()]),
        "burst_types": ", ".join(d.params.get("burst_types", [])),
        "description": d.description,
    }
    for d in datasets
])

## Run Both Detectors and Quantify Results

This section runs the new iterative detector and the traditional participation-based detector on the same synthetic datasets. Edit `iterative_config` or `traditional_config` here when testing parameter sensitivity.

The Poisson-only datasets are negative controls: any network-burst calls there should be treated as false positives to investigate.

Quantification includes event-level matching against ground truth plus summaries of detector output columns. A predicted event is counted as matched when it overlaps an unmatched truth burst by at least `MIN_OVERLAP_S`.

In [ ]:
iterative_config = IterativeBurstConfig()
traditional_config = BurstDetectorConfig()
MIN_OVERLAP_S = 0.03


def interval_overlap(a: tuple[float, float], b: tuple[float, float]) -> float:
    return max(0.0, min(a[1], b[1]) - max(a[0], b[0]))


def interval_union(a: tuple[float, float], b: tuple[float, float]) -> float:
    return max(a[1], b[1]) - min(a[0], b[0])


def match_events_to_truth(events: pd.DataFrame, truth: list[tuple[float, float]], min_overlap_s: float = MIN_OVERLAP_S) -> list[dict]:
    if events.empty or not truth:
        return []
    event_intervals = list(zip(events["start"].astype(float), events["end"].astype(float)))
    candidates = []
    for event_idx, event_interval in enumerate(event_intervals):
        for truth_idx, truth_interval in enumerate(truth):
            overlap = interval_overlap(event_interval, truth_interval)
            if overlap >= min_overlap_s:
                candidates.append((overlap, event_idx, truth_idx, event_interval, truth_interval))
    candidates.sort(reverse=True, key=lambda x: x[0])

    used_events = set()
    used_truth = set()
    matches = []
    for overlap, event_idx, truth_idx, event_interval, truth_interval in candidates:
        if event_idx in used_events or truth_idx in used_truth:
            continue
        used_events.add(event_idx)
        used_truth.add(truth_idx)
        union = interval_union(event_interval, truth_interval)
        matches.append({
            "event_idx": event_idx,
            "truth_idx": truth_idx,
            "overlap_s": overlap,
            "iou": overlap / union if union > 0 else 0.0,
            "onset_error_s": event_interval[0] - truth_interval[0],
            "offset_error_s": event_interval[1] - truth_interval[1],
            "duration_error_s": (event_interval[1] - event_interval[0]) - (truth_interval[1] - truth_interval[0]),
        })
    return matches


def summarize_event_quality(events: pd.DataFrame) -> dict:
    quality_cols = [
        "duration_s", "peak_synchrony", "participation", "total_spikes",
        "llr_aggregate", "composite_peak", "composite_mean", "ff_peak",
    ]
    if events.empty:
        return {f"{col}_median": 0.0 for col in quality_cols} | {f"{col}_mean": 0.0 for col in quality_cols}
    summary = {}
    for col in quality_cols:
        if col in events:
            summary[f"{col}_median"] = float(events[col].median())
            summary[f"{col}_mean"] = float(events[col].mean())
    return summary


def interval_total_duration(intervals: list[tuple[float, float]]) -> float:
    return float(sum(max(0.0, e - s) for s, e in intervals))


def score_events_against_truth(events: pd.DataFrame, truth: list[tuple[float, float]], duration_s: float) -> dict:
    event_intervals = [] if events.empty else list(zip(events["start"].astype(float), events["end"].astype(float)))
    matches = match_events_to_truth(events, truth)
    matched_event_indices = {m["event_idx"] for m in matches}
    matched_truth_indices = {m["truth_idx"] for m in matches}
    true_positive = len(matches)
    false_positive = len(event_intervals) - true_positive
    false_negative = len(truth) - true_positive
    precision = true_positive / len(event_intervals) if event_intervals else (1.0 if not truth else 0.0)
    recall = true_positive / len(truth) if truth else (1.0 if not event_intervals else 0.0)
    f1 = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0.0

    truth_time_s = interval_total_duration(truth)
    predicted_time_s = interval_total_duration(event_intervals)
    matched_overlap_s = float(sum(m["overlap_s"] for m in matches))
    false_positive_time_s = sum(
        (e - s) for idx, (s, e) in enumerate(event_intervals)
        if idx not in matched_event_indices
    )
    missed_truth_time_s = sum(
        (e - s) for idx, (s, e) in enumerate(truth)
        if idx not in matched_truth_indices
    )
    coverage_recall = matched_overlap_s / truth_time_s if truth_time_s > 0 else (1.0 if predicted_time_s == 0 else 0.0)
    coverage_precision = matched_overlap_s / predicted_time_s if predicted_time_s > 0 else (1.0 if truth_time_s == 0 else 0.0)
    coverage_f1 = (
        2 * coverage_precision * coverage_recall / (coverage_precision + coverage_recall)
        if coverage_precision + coverage_recall > 0 else 0.0
    )
    count_error = len(event_intervals) - len(truth)
    count_abs_error = abs(count_error)
    count_ratio = len(event_intervals) / len(truth) if truth else (0.0 if not event_intervals else float("inf"))
    event_rate_per_min = len(event_intervals) / (duration_s / 60.0) if duration_s > 0 else 0.0
    truth_rate_per_min = len(truth) / (duration_s / 60.0) if duration_s > 0 else 0.0
    false_positive_rate_per_min = false_positive / (duration_s / 60.0) if duration_s > 0 else 0.0
    missed_rate = false_negative / len(truth) if truth else 0.0
    fragmentation_index = len(event_intervals) / true_positive if true_positive > 0 else 0.0
    merge_flag = any(m["duration_error_s"] > 0.5 for m in matches)
    split_or_miss_flag = false_negative > 0 or fragmentation_index > 1.5

    return {
        "truth_bursts": len(truth),
        "detected_network_bursts": len(event_intervals),
        "true_positive_events": true_positive,
        "false_positive_events": false_positive,
        "false_negative_events": false_negative,
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "mean_iou": float(np.mean([m["iou"] for m in matches])) if matches else 0.0,
        "median_abs_onset_error_s": float(np.median(np.abs([m["onset_error_s"] for m in matches]))) if matches else 0.0,
        "median_abs_offset_error_s": float(np.median(np.abs([m["offset_error_s"] for m in matches]))) if matches else 0.0,
        "median_abs_duration_error_s": float(np.median(np.abs([m["duration_error_s"] for m in matches]))) if matches else 0.0,
        "truth_time_s": truth_time_s,
        "predicted_time_s": predicted_time_s,
        "matched_overlap_s": matched_overlap_s,
        "coverage_precision": coverage_precision,
        "coverage_recall": coverage_recall,
        "coverage_f1": coverage_f1,
        "false_positive_time_s": false_positive_time_s,
        "missed_truth_time_s": missed_truth_time_s,
        "count_error": count_error,
        "count_abs_error": count_abs_error,
        "count_ratio": count_ratio,
        "truth_rate_per_min": truth_rate_per_min,
        "event_rate_per_min": event_rate_per_min,
        "false_positive_rate_per_min": false_positive_rate_per_min,
        "missed_rate": missed_rate,
        "fragmentation_index": fragmentation_index,
        "merge_flag": merge_flag,
        "split_or_miss_flag": split_or_miss_flag,
    }


def build_event_quality_table(dataset: SyntheticDataset, result) -> pd.DataFrame:
    events = result.network_bursts.copy()
    if events.empty:
        return pd.DataFrame()
    matches = match_events_to_truth(events, dataset.burst_intervals)
    matched_event_indices = {m["event_idx"] for m in matches}
    match_by_event = {m["event_idx"]: m for m in matches}
    rows = []
    for idx, row in events.reset_index(drop=True).iterrows():
        match = match_by_event.get(idx, {})
        rows.append({
            "dataset": dataset.name,
            "event_idx": idx,
            "is_true_positive": idx in matched_event_indices,
            "start": float(row.start),
            "end": float(row.end),
            "duration_s": float(row.duration_s),
            "truth_idx": match.get("truth_idx", np.nan),
            "iou": match.get("iou", 0.0),
            "onset_error_s": match.get("onset_error_s", np.nan),
            "duration_error_s": match.get("duration_error_s", np.nan),
            "peak_synchrony": float(row.get("peak_synchrony", np.nan)),
            "participation": float(row.get("participation", np.nan)),
            "llr_aggregate": float(row.get("llr_aggregate", np.nan)),
            "composite_peak": float(row.get("composite_peak", np.nan)),
            "composite_mean": float(row.get("composite_mean", np.nan)),
            "ff_peak": float(row.get("ff_peak", np.nan)),
        })
    return pd.DataFrame(rows)


def run_detector_method(dataset: SyntheticDataset, method: str, config):
    if method == "iterative":
        return compute_iterative_bursts(dataset.spike_times, config=config)
    if method == "traditional":
        return compute_network_bursts(dataset.spike_times, config=config)
    raise ValueError(f"Unknown detector method: {method}")


def run_all_detectors(
    datasets: list[SyntheticDataset],
    detector_configs: dict[str, object],
) -> tuple[dict, pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    results = {}
    score_rows = []
    quality_rows = []
    event_quality_tables = []
    for method, config in detector_configs.items():
        results[method] = {}
        for dataset in datasets:
            try:
                result = run_detector_method(dataset, method, config)
                results[method][dataset.name] = result
                score = score_events_against_truth(result.network_bursts, dataset.burst_intervals, dataset.duration_s)
                quality = summarize_event_quality(result.network_bursts)
                negative_control = len(dataset.burst_intervals) == 0
                score_rows.append({
                    "method": method,
                    "dataset": dataset.name,
                    "type": dataset.culture_type,
                    "expected": "negative control" if negative_control else "burst-positive",
                    **score,
                    "negative_control_overcalled": negative_control and score["detected_network_bursts"] > 0,
                    "burst_truth_missed": (not negative_control) and score["false_negative_events"] > 0,
                    "burstlets": len(result.burstlets),
                    "superbursts": len(result.superbursts),
                    "iterations": result.diagnostics.get("n_iterations", np.nan),
                    "convergence_delta": result.diagnostics.get("convergence_delta", np.nan),
                    "adaptive_bin_ms": result.diagnostics.get("adaptive_bin_ms", np.nan),
                    "composite_threshold": result.diagnostics.get("composite_threshold", np.nan),
                    "participation_threshold": result.plot_data.get("participation_threshold", np.nan),
                })
                quality_rows.append({"method": method, "dataset": dataset.name, **quality})
                event_quality_tables.append(build_event_quality_table(dataset, result).assign(method=method))
            except (IterativeBurstError, BurstDetectorError) as exc:
                score_rows.append({
                    "method": method,
                    "dataset": dataset.name,
                    "type": dataset.culture_type,
                    "error": str(exc),
                })
    event_quality = pd.concat(event_quality_tables, ignore_index=True) if event_quality_tables else pd.DataFrame()
    return results, pd.DataFrame(score_rows), pd.DataFrame(quality_rows), event_quality


detector_configs = {
    "iterative": iterative_config,
    "traditional": traditional_config,
}
results, detection_summary, quality_summary, event_quality = run_all_detectors(datasets, detector_configs)
detection_summary

## Quantification of Detector Quality Metrics

The evaluation compares two methods: **iterative** and **traditional**. It separates three questions:

- **Event count quality:** precision, recall, F1, count error, false-positive event rate, and missed-event rate.
- **Temporal quality:** matched-event IoU, coverage precision/recall/F1, onset/offset/duration error, false-positive time, and missed truth time.
- **Detector signal quality:** event-level LLR/composite/FF where available. Traditional detector rows will have zeros or missing values for iterative-only quality columns.

`detection_summary` is one row per dataset and method. `quality_summary` aggregates detector output columns per dataset and method. `event_quality` keeps one row per predicted network burst with true-positive status and timing errors when matched to ground truth.

In [ ]:
display_cols = [
    "method", "dataset", "precision", "recall", "f1", "coverage_f1", "mean_iou",
    "count_error", "false_positive_rate_per_min", "missed_rate",
    "median_abs_onset_error_s", "median_abs_duration_error_s",
]
detection_summary[display_cols]

## Metric Plots

The first figure summarizes detection accuracy and detector quality metrics across all synthetic datasets. The second figure emphasizes evaluation failure modes: count error, temporal coverage, missed truth time, and quality-metric separation between true and false predicted events. Figures are saved into `FIGURE_OUTPUT_DIR`.

In [ ]:
def save_metric_figure(fig, filename: str) -> Path:
    path = FIGURE_OUTPUT_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    return path


METHOD_COLORS = {"iterative": "#2563eb", "traditional": "#f97316"}
METHOD_OFFSETS = {"iterative": -0.18, "traditional": 0.18}


def ordered_metric_frame(detection_summary: pd.DataFrame, quality_summary: pd.DataFrame) -> pd.DataFrame:
    plot_df = detection_summary.merge(quality_summary, on=["method", "dataset"], how="left")
    return plot_df.sort_values(["expected", "dataset", "method"], ascending=[True, True, True]).reset_index(drop=True)


def method_pivot(df: pd.DataFrame, value: str) -> pd.DataFrame:
    return df.pivot(index="dataset", columns="method", values=value)


def plot_metric_summary(detection_summary: pd.DataFrame, quality_summary: pd.DataFrame) -> None:
    plot_df = ordered_metric_frame(detection_summary, quality_summary)
    dataset_order = plot_df.drop_duplicates("dataset")["dataset"].tolist()
    labels = [d.replace("-", "\n") for d in dataset_order]
    x = np.arange(len(dataset_order))

    fig, axes = plt.subplots(3, 2, figsize=(16, 11), sharex=True)
    axes = axes.ravel()

    for metric, ax, title, ylabel, ylim in [
        ("f1", axes[0], "Event F1", "score", (0, 1.05)),
        ("coverage_f1", axes[1], "Temporal coverage F1", "score", (0, 1.05)),
        ("mean_iou", axes[2], "Mean matched IoU", "IoU", (0, 1.05)),
        ("count_error", axes[3], "Count error", "detected - truth", None),
        ("false_positive_time_s", axes[4], "False-positive predicted time", "seconds", None),
        ("missed_truth_time_s", axes[5], "Missed truth time", "seconds", None),
    ]:
        pivot = method_pivot(plot_df, metric).reindex(dataset_order)
        for method in pivot.columns:
            ax.bar(
                x + METHOD_OFFSETS.get(method, 0.0), pivot[method], width=0.32,
                color=METHOD_COLORS.get(method, "#64748b"), alpha=0.85, label=method,
            )
        ax.set_title(title)
        ax.set_ylabel(ylabel)
        if ylim:
            ax.set_ylim(*ylim)
        ax.grid(axis="y", color="#e5e7eb", linewidth=0.7)
        ax.set_axisbelow(True)
        ax.legend(frameon=False, fontsize=8)

    for ax in axes[-2:]:
        ax.set_xticks(x)
        ax.set_xticklabels(labels, rotation=0, ha="center", fontsize=7)

    fig.suptitle("Iterative vs traditional detector: evaluation metrics", y=1.01)
    plt.tight_layout()
    saved_path = save_metric_figure(fig, "all-all-method-comparison-summary.png")
    print(f"Saved method comparison summary figure: {saved_path}")


def plot_evaluation_dashboard(detection_summary: pd.DataFrame, quality_summary: pd.DataFrame, event_quality: pd.DataFrame) -> None:
    plot_df = ordered_metric_frame(detection_summary, quality_summary)
    methods = plot_df["method"].drop_duplicates().tolist()
    dataset_order = plot_df.drop_duplicates("dataset")["dataset"].tolist()

    fig, axes = plt.subplots(2, 2, figsize=(16, 10))

    heat_cols = ["precision", "recall", "f1", "coverage_precision", "coverage_recall", "coverage_f1", "mean_iou"]
    heat_rows = []
    heat_labels = []
    for dataset in dataset_order:
        for method in methods:
            row = plot_df[(plot_df["dataset"] == dataset) & (plot_df["method"] == method)]
            if row.empty:
                continue
            heat_rows.append(row[heat_cols].iloc[0].to_numpy(dtype=float))
            heat_labels.append(f"{dataset} [{method}]")
    heat = np.vstack(heat_rows) if heat_rows else np.zeros((0, len(heat_cols)))
    im = axes[0, 0].imshow(heat, aspect="auto", vmin=0, vmax=1, cmap="viridis")
    axes[0, 0].set_yticks(np.arange(len(heat_labels)))
    axes[0, 0].set_yticklabels(heat_labels, fontsize=6)
    axes[0, 0].set_xticks(np.arange(len(heat_cols)))
    axes[0, 0].set_xticklabels(heat_cols, rotation=35, ha="right", fontsize=8)
    axes[0, 0].set_title("Score heatmap")
    fig.colorbar(im, ax=axes[0, 0], fraction=0.046, pad=0.02)

    f1_pivot = method_pivot(plot_df, "f1").reindex(dataset_order)
    if {"iterative", "traditional"}.issubset(f1_pivot.columns):
        delta_f1 = f1_pivot["iterative"] - f1_pivot["traditional"]
        axes[0, 1].axhline(0, color="#111827", linewidth=0.8)
        axes[0, 1].bar(np.arange(len(delta_f1)), delta_f1, color=np.where(delta_f1 >= 0, "#2563eb", "#f97316"), alpha=0.85)
        axes[0, 1].set_xticks(np.arange(len(delta_f1)))
        axes[0, 1].set_xticklabels([d.replace("-", "\n") for d in dataset_order], fontsize=7)
        axes[0, 1].set_ylabel("iterative - traditional")
        axes[0, 1].set_title("F1 delta by dataset")
        axes[0, 1].grid(axis="y", color="#e5e7eb")

    for method, group in plot_df.groupby("method"):
        axes[1, 0].scatter(group["truth_time_s"], group["predicted_time_s"],
                           c=METHOD_COLORS.get(method, "#64748b"), s=45, alpha=0.80, label=method)
    lim = max(float(plot_df[["truth_time_s", "predicted_time_s"]].to_numpy().max()), 1.0)
    axes[1, 0].plot([0, lim], [0, lim], color="#111827", linestyle="--", linewidth=0.9)
    axes[1, 0].set_xlabel("truth burst time (s)")
    axes[1, 0].set_ylabel("predicted burst time (s)")
    axes[1, 0].set_title("Total predicted vs truth burst time")
    axes[1, 0].legend(frameon=False, fontsize=8)

    axes[1, 1].set_title("Event quality by method and match status")
    if not event_quality.empty:
        for method, marker in [("iterative", "o"), ("traditional", "s")]:
            subset = event_quality[event_quality["method"] == method]
            if subset.empty:
                continue
            colors = np.where(subset["is_true_positive"], METHOD_COLORS.get(method, "#64748b"), "#dc2626")
            axes[1, 1].scatter(subset["participation"], subset["peak_synchrony"],
                               c=colors, marker=marker, s=35, alpha=0.75, edgecolor="white", linewidth=0.4,
                               label=method)
        axes[1, 1].set_xlabel("event participation")
        axes[1, 1].set_ylabel("peak synchrony")
        axes[1, 1].legend(frameon=False, fontsize=8)
    else:
        axes[1, 1].text(0.5, 0.5, "No predicted events", ha="center", va="center")

    plt.tight_layout()
    saved_path = save_metric_figure(fig, "all-all-method-comparison-dashboard.png")
    print(f"Saved method comparison dashboard figure: {saved_path}")


plot_metric_summary(detection_summary, quality_summary)
plot_evaluation_dashboard(detection_summary, quality_summary, event_quality)

## All-Dataset Overview

Ground-truth bursts are filled green bands. Ground-truth silence periods are filled blue bands. Predicted iterative network bursts are red hatched outlines. Traditional detector predictions are shown separately in the method-comparison metric plots and can be inspected by changing `METHOD_FOR_RASTER` below. The trace below each raster is the selected detector's composite or participation signal, normalized per dataset for visual comparison. The figure is saved into `FIGURE_OUTPUT_DIR`.

In [ ]:
from matplotlib.patches import Patch


TRUTH_BURST_STYLE = {"facecolor": "#22c55e", "alpha": 0.22, "edgecolor": "none"}
TRUTH_SILENCE_STYLE = {"facecolor": "#38bdf8", "alpha": 0.22, "edgecolor": "none"}
PREDICTED_BURST_STYLE = {
    "facecolor": "none",
    "edgecolor": "#dc2626",
    "alpha": 0.95,
    "linewidth": 1.4,
    "hatch": "////",
}
METHOD_FOR_RASTER = "iterative"

LEGEND_HANDLES = [
    Patch(label="truth burst", **TRUTH_BURST_STYLE),
    Patch(label="truth silence", **TRUTH_SILENCE_STYLE),
    Patch(label="predicted network", **PREDICTED_BURST_STYLE),
]


def save_figure(fig, filename: str) -> Path:
    path = FIGURE_OUTPUT_DIR / filename
    fig.savefig(path, dpi=200, bbox_inches="tight")
    return path


def add_truth_spans(ax, dataset: SyntheticDataset) -> None:
    for s, e in dataset.burst_intervals:
        ax.axvspan(s, e, zorder=0, **TRUTH_BURST_STYLE)
    for s, e in dataset.silence_intervals:
        ax.axvspan(s, e, zorder=0, **TRUTH_SILENCE_STYLE)


def add_detected_spans(ax, events: pd.DataFrame) -> None:
    if events.empty:
        return
    for _, row in events.reset_index(drop=True).iterrows():
        ax.axvspan(float(row.start), float(row.end), zorder=3, **PREDICTED_BURST_STYLE)


def plot_overview(datasets: list[SyntheticDataset], results: dict, method: str = METHOD_FOR_RASTER) -> None:
    fig, axes = plt.subplots(len(datasets), 1, figsize=(12, 1.55 * len(datasets)), sharex=True)
    if len(datasets) == 1:
        axes = [axes]
    for ax, dataset in zip(axes, datasets):
        result = results.get(method, {}).get(dataset.name)
        add_truth_spans(ax, dataset)
        if result is not None:
            add_detected_spans(ax, result.network_bursts)
            t = result.plot_data["t"]
            signal_key = "composite_signal" if "composite_signal" in result.plot_data else "participation_signal"
            signal = np.asarray(result.plot_data[signal_key], dtype=float)
            span = np.nanmax(signal) - np.nanmin(signal)
            y = (signal - np.nanmin(signal)) / span if span > 1e-12 else signal * 0.0
            ax.plot(t, y, color="#111827", lw=0.9, zorder=2)
        ax.set_ylim(-0.05, 1.05)
        ax.set_ylabel(dataset.name, rotation=0, ha="right", va="center", fontsize=8)
    axes[0].legend(handles=LEGEND_HANDLES, loc="upper right", ncols=3, fontsize=8, frameon=False)
    axes[-1].set_xlabel("time (s)")
    fig.suptitle(f"{method} detector overview with ground truth and predicted network bursts", y=1.002)
    plt.tight_layout()
    saved_path = save_figure(fig, f"all-all-{method}-overview.png")
    print(f"Saved overview figure: {saved_path}")


plot_overview(datasets, results, METHOD_FOR_RASTER)

## Detailed Visual Checks

The detailed view combines spike rasters, ground-truth intervals, detector event spans, and the detector's internal signals. Truth uses filled bands; predictions use red hatched outlines. Change `datasets_to_plot` to inspect other cases. Each detailed figure is saved into `FIGURE_OUTPUT_DIR`.


In [ ]:
def plot_dataset_detail(dataset: SyntheticDataset, result, method: str = METHOD_FOR_RASTER) -> None:
    n_trace_axes = 5 if method == "iterative" else 2
    fig, axes = plt.subplots(
        1 + n_trace_axes, 1, figsize=(13, 4.5 + 1.2 * n_trace_axes), sharex=True,
        gridspec_kw={"height_ratios": [2.6] + [1] * n_trace_axes},
    )
    ax_raster = axes[0]
    add_truth_spans(ax_raster, dataset)
    add_detected_spans(ax_raster, result.network_bursts)

    for y, unit in enumerate(dataset.spike_times):
        spikes = dataset.spike_times[unit]
        ax_raster.scatter(spikes, np.full_like(spikes, y), s=3, color="#111827", linewidths=0)
    ax_raster.set_ylabel("unit")
    ax_raster.set_title(f"{dataset.name}: {dataset.description}", loc="left", fontsize=10)
    ax_raster.legend(handles=LEGEND_HANDLES, loc="upper right", ncols=3, fontsize=8, frameon=False)

    t = result.plot_data["t"]
    traces = [
        ("participation_signal", "participation", "#2563eb"),
        ("rate_signal", "rate", "#111827"),
    ]
    if "composite_signal" in result.plot_data:
        traces.extend([
            ("composite_signal", "composite", "#dc2626"),
            ("llr_signal", "LLR", "#7c3aed"),
            ("ff_signal", "Fano factor", "#059669"),
        ])
    for ax, (key, label, color) in zip(axes[1:], traces):
        add_truth_spans(ax, dataset)
        add_detected_spans(ax, result.network_bursts)
        ax.plot(t, result.plot_data[key], color=color, lw=1.0)
        if key == "composite_signal" and "composite_threshold" in result.diagnostics:
            ax.axhline(result.diagnostics["composite_threshold"], color="#991b1b", ls="--", lw=0.9)
        if key == "participation_signal" and "participation_threshold" in result.plot_data:
            ax.axhline(result.plot_data["participation_threshold"], color="#1d4ed8", ls="--", lw=0.9)
        ax.set_ylabel(label)

    axes[-1].set_xlabel("time (s)")
    plt.tight_layout()
    saved_path = save_figure(fig, f"{method}-{dataset.name}.png")
    print(f"Saved detail figure: {saved_path}")


datasets_by_name = {d.name: d for d in datasets}
datasets_to_plot = [
    "random-low-independent",
    "random-medium-many-units",
    "network-medium-short-duration",
    "network-medium-variable-duration",
    "network-medium-randomized-cascade-a",
    "network-medium-clustered-bursts",
    "composite-mixed-many-burst-types",
    "network-medium-post-burst-silence",
]

for name in datasets_to_plot:
    plot_dataset_detail(datasets_by_name[name], results[METHOD_FOR_RASTER][name], METHOD_FOR_RASTER)

## Event Tables

Use these tables to inspect timing and quality columns for any dataset. `network_bursts` is usually the first table to inspect because it is the main per-culture burst call.


In [ ]:
METHOD_FOR_TABLE = "iterative"
DATASET_FOR_TABLE = "network-medium-post-burst-silence"
selected = results[METHOD_FOR_TABLE][DATASET_FOR_TABLE]

print(f"Diagnostics for {METHOD_FOR_TABLE} / {DATASET_FOR_TABLE}")
for key in [
    "n_iterations", "convergence_delta", "adaptive_bin_ms", "biological_isi_s",
    "composite_baseline", "composite_threshold", "burstlet_merge_gap_s", "network_merge_gap_s",
]:
    if key in selected.diagnostics:
        print(f"{key}: {selected.diagnostics[key]}")

if "feature_weights_final" in selected.diagnostics:
    print("\nFinal feature weights [PFR, participation, FF..., LLR, burstiness]:")
    print(np.round(selected.diagnostics["feature_weights_final"], 3))

selected.network_bursts

## Interpretation Notes

- The notebook now compares `iterative` and `traditional` detectors on the same synthetic spike trains.
- Independent Poisson datasets are negative controls. If `negative_control_overcalled` is `True`, that method is producing false positives on unconnected spike trains.
- `detection_summary` reports event-count quality, temporal quality, and failure-mode metrics per method and dataset. Use event F1 for count matching and coverage F1 / IoU for temporal overlap.
- `count_error`, `fragmentation_index`, and `merge_flag` help distinguish over-splitting from over-merging.
- `quality_summary` quantifies detector-specific event metrics per method. Iterative-only columns such as composite and LLR are absent or zero for the traditional detector.
- Set `METHOD_FOR_RASTER` or `METHOD_FOR_TABLE` to `"traditional"` to inspect traditional detector rasters or event tables.